In [1]:
# Install required packages

!pip install -q -U langgraph langchain-core langchain-openai openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 47.0 MB/s eta 0:00:00


In [3]:
# Import required libraries

import os

from typing import Annotated, TypedDict

from google.colab import userdata

# LangChain message classes
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    SystemMessage,
    AIMessage
)

# Tool decorator
from langchain_core.tools import tool

# OpenRouter through OpenAI-compatible LangChain interface
from langchain_openai import ChatOpenAI

# LangGraph
from langgraph.graph import StateGraph, START, END

# Used to add messages to graph state
from langgraph.graph.message import add_messages

In [4]:
# ============================================================
# 1. OPENROUTER API KEY & MODEL SETUP
# ============================================================

# Get OpenRouter API key from Colab Secrets

OPENROUTER_API_KEY = userdata.get(
    "PDF_Chatbot"
)


# Check whether key was found

if not OPENROUTER_API_KEY:

    raise RuntimeError(
        "Please add 'PDF_Chatbot' to Colab Secrets."
    )


print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [6]:
# Create OpenRouter LLM

llm = ChatOpenAI(

    # OpenRouter API endpoint
    base_url="https://openrouter.ai/api/v1",

    # Your OpenRouter key
    api_key=OPENROUTER_API_KEY,

    # OpenRouter model
    model="openai/gpt-oss-20b",

    # Consistent responses
    temperature=0
)


print("OpenRouter LLM initialized successfully.")

OpenRouter LLM initialized successfully.


In [7]:
# ============================================================
# HELPER FUNCTION
# ============================================================

def extract_text_safely(content) -> str:

    # If content is already a string
    if isinstance(content, str):

        return content


    # If content is a list
    elif isinstance(content, list):

        parts = []

        for item in content:

            if isinstance(item, dict) and "text" in item:

                parts.append(
                    item["text"]
                )

            elif isinstance(item, str):

                parts.append(item)

            else:

                parts.append(
                    str(item)
                )

        return "".join(parts)


    # Convert other types to string
    return str(content)

In [8]:
# ============================================================
# 2. STATE DEFINITION
# ============================================================

class AgentState(TypedDict):

    # Stores conversation messages
    messages: Annotated[
        list[BaseMessage],
        add_messages
    ]

    # Stores the next node to execute
    next_node: str

In [9]:
# ============================================================
# 3. TOOL DEFINITION
# ============================================================

@tool
def process_refund(
    user_id: str,
    amount: float
) -> str:

    """
    Processes a refund for a user.
    """

    return (
        f"SUCCESS: Refund of ${amount} "
        f"has been processed for User "
        f"'{user_id}'."
    )

In [10]:
# ============================================================
# 4. SUPERVISOR AGENT
# ============================================================

def supervisor_agent(
    state: AgentState
) -> AgentState:

    # Instructions for supervisor

    system_prompt = (
        "You are a Support Router.\n"

        "Analyze the user's request.\n"

        "- If it is general technical troubleshooting, "
        "respond with 'SUPPORT'.\n"

        "- If it involves financial refunds or "
        "account modifications, respond with 'ACTION'.\n"

        "Respond ONLY with SUPPORT or ACTION."
    )


    # Combine system prompt with user messages

    messages = [
        SystemMessage(
            content=system_prompt
        )
    ] + state["messages"]


    # Ask OpenRouter LLM

    response = llm.invoke(
        messages
    )


    # Extract response text

    raw_text = extract_text_safely(
        response.content
    )


    # Convert to uppercase

    decision = raw_text.strip().upper()


    # Decide next agent

    if "ACTION" in decision:

        next_step = (
            "account_actions_agent"
        )

    elif "SUPPORT" in decision:

        next_step = (
            "tech_support_agent"
        )

    else:

        next_step = END


    # Return next node

    return {
        "next_node": next_step
    }

In [11]:
# ============================================================
# 5. TECHNICAL SUPPORT AGENT
# ============================================================

def tech_support_agent(
    state: AgentState
) -> AgentState:

    # Instructions for technical support

    system_prompt = SystemMessage(
        content=(
            "You are a helpful Technical Support "
            "Specialist. "

            "Provide clear and concise "
            "troubleshooting guidance."
        )
    )


    # Combine instructions and user messages

    messages = [
        system_prompt
    ] + state["messages"]


    # Ask OpenRouter

    response = llm.invoke(
        messages
    )


    # Extract answer

    text_content = extract_text_safely(
        response.content
    )


    # Return answer

    return {

        "messages": [
            AIMessage(
                content=
                f"[Tech Support]: "
                f"{text_content}"
            )
        ],

        "next_node": END
    }

In [12]:
# ============================================================
# 6. ACCOUNT ACTION AGENT
# ============================================================

def account_actions_agent(
    state: AgentState
) -> AgentState:

    # Give refund tool to the LLM

    llm_with_tools = llm.bind_tools(
        [process_refund]
    )


    # Instructions for account agent

    system_prompt = SystemMessage(
        content=(
            "You are an Account Manager. "

            "Use the process_refund tool "
            "when the user requests a refund."
        )
    )


    # Combine system message and user messages

    messages = [
        system_prompt
    ] + state["messages"]


    # Ask OpenRouter

    response = llm_with_tools.invoke(
        messages
    )


    # Check whether the LLM requested a tool

    if response.tool_calls:

        # Get first tool call

        tool_call = response.tool_calls[0]


        # Execute refund tool

        tool_output = process_refund.invoke(
            tool_call["args"]
        )


        # Create final response

        final_msg = (
            "[Account Agent]: "
            "Executed tool. Result: "
            f"{tool_output}"
        )


    else:

        # If no tool was called

        text_content = extract_text_safely(
            response.content
        )


        final_msg = (
            "[Account Agent]: "
            f"{text_content}"
        )


    # Return final state

    return {

        "messages": [
            AIMessage(
                content=final_msg
            )
        ],

        "next_node": END
    }

In [13]:
# ============================================================
# 7. CONSTRUCT LANGGRAPH WORKFLOW
# ============================================================

# Create StateGraph

workflow = StateGraph(
    AgentState
)


# Add Supervisor node

workflow.add_node(
    "supervisor",
    supervisor_agent
)


# Add Technical Support node

workflow.add_node(
    "tech_support_agent",
    tech_support_agent
)


# Add Account Action node

workflow.add_node(
    "account_actions_agent",
    account_actions_agent
)

In [14]:
# START → SUPERVISOR

workflow.add_edge(
    START,
    "supervisor"
)

In [15]:
# ============================================================
# CONDITIONAL ROUTING
# ============================================================

workflow.add_conditional_edges(

    # Start routing from supervisor

    "supervisor",

    # Read next_node from state

    lambda state:
        state["next_node"],

    {

        # Technical question
        "tech_support_agent":
            "tech_support_agent",

        # Refund/account question
        "account_actions_agent":
            "account_actions_agent",

        # Finish
        END:
            END
    }
)

In [16]:
# Technical Support → END

workflow.add_edge(
    "tech_support_agent",
    END
)


# Account Action → END

workflow.add_edge(
    "account_actions_agent",
    END
)

In [17]:
# ============================================================
# COMPILE GRAPH
# ============================================================

app = workflow.compile()

print(
    "LangGraph workflow compiled successfully."
)

LangGraph workflow compiled successfully.


In [18]:
# ============================================================
# 8. RUN DEMO
# ============================================================

def run_demo(
    user_query: str
):

    # Display user query

    print(
        "\n================ USER QUERY ================"
    )

    print(
        user_query
    )


    # Create initial graph input

    inputs = {

        "messages": [
            HumanMessage(
                content=user_query
            )
        ]

    }


    # Execute LangGraph

    result = app.invoke(
        inputs
    )


    # Display final answer

    print(
        "\n================ SYSTEM RESPONSE ================"
    )

    print(
        result["messages"][-1].content
    )

In [19]:
# ============================================================
# TEST CASE 1
# ============================================================

run_demo(
    "My app keeps freezing whenever I try "
    "to upload a PNG file. How can I fix this?"
)


================ USER QUERY ================
My app keeps freezing whenever I try to upload a PNG file. How can I fix this?

================ SYSTEM RESPONSE ================
[Tech Support]: Below is a quick‑start checklist you can follow to isolate and fix the “app freezes when uploading a PNG” issue.  
Feel free to skip any steps that don’t apply to your environment.

---

## 1. Gather Basic Information  
| What to check | Why it matters | How to check |
|---------------|----------------|--------------|
| **App version** | Bugs are often version‑specific. | Settings → About / Help → Version |
| **OS & build** | Compatibility issues can cause hangs. | Settings → About phone / System → OS version |
| **Device / browser** | Some devices or browsers handle PNGs differently. | Note the device model or browser name/number |
| **File size & dimensions** | Large images can exhaust memory. | Right‑click → Properties (Windows) / Get Info (macOS) |
| **Error logs** | May contain stack traces o

In [20]:
# ============================================================
# TEST CASE 2
# ============================================================

run_demo(
    "I was billed twice by mistake. "
    "Please refund $49.99 for my account "
    "'user_9876'."
)


================ USER QUERY ================
I was billed twice by mistake. Please refund $49.99 for my account 'user_9876'.

================ SYSTEM RESPONSE ================
[Account Agent]: Executed tool. Result: SUCCESS: Refund of $49.99 has been processed for User 'user_9876'.
